# Retail Transaction Analytics Pipeline
## Medallion Architecture: Bronze → Silver → Gold

**Purpose:** End-to-end data pipeline for retail transaction analysis

**Data Source:** Azure Data Lake Storage (ADLS) - Raw parquet files

**Pipeline Stages:**
* **Bronze Layer:** Raw data ingestion from ADLS mounted storage
* **Silver Layer:** Data quality checks, cleaning, and standardization
* **Gold Layer:** Business metrics and analytics views

**Data Fields:**
* TransactionID, CustomerID, TransactionDate
* Quantity, Amount, Discount
* PaymentType, StoreRegion, DeviceUsed
* StoreLocation, CustomerLoyaltyLevel, ProductCategory

## 🟤 Bronze Layer - Raw Data Ingestion

**Objective:** Mount Azure Data Lake Storage and load raw transaction data

**Steps:**
1. Mount ADLS container to `/mnt/retail` using access key authentication
2. Explore mounted directory structure
3. Read raw parquet files from bronze location
4. Load data without any transformations (raw state)

**Data Format:** Parquet

**Storage Path:** `/mnt/retail/bronze/nagarjuna040596/AZURE-DATA-ENGINEER/refs/heads/main/`

In [0]:
dbutils.fs.mount(
  source = "wasbs://CONTAINER@STORAGENAME.blob.core.windows.net",
  mount_point = "/mnt/retail",
  extra_configs = {"fs.azure.account.key.STORAGENAME.blob.core.windows.net":"PASSYORACCESSKEY"})



In [0]:
dbutils.fs.ls('/mnt/retail/')

In [0]:
dbutils.fs.ls('/mnt/retail/bronze/nagarjuna040596/AZURE-DATA-ENGINEER/refs/heads/main/')

In [0]:
from pyspark.sql.functions import col, upper, trim, when
bronze_df = spark.read.parquet("/mnt/retail/bronze/nagarjuna040596/AZURE-DATA-ENGINEER/refs/heads/main/")
display(bronze_df)


## ⚪ Silver Layer - Data Cleaning & Standardization

**Objective:** Clean, validate, and standardize raw data for reliable analytics

### Null Handling
Filter out records with missing critical fields (TransactionID, CustomerID, TransactionDate)

In [0]:
df_clean1 = bronze_df.filter(
    (col("TransactionID").isNotNull()) &
    (col("CustomerID").isNotNull()) &
    (col("TransactionDate").isNotNull())
)
display(df_clean1)

### Text Standardization
Convert PaymentType, StoreRegion, DeviceUsed to UPPERCASE and trim whitespace

In [0]:
# Trim and standardize text fields
df_clean2 = df_clean1.withColumn("PaymentType", upper(trim(col("PaymentType")))) \
                     .withColumn("StoreRegion", upper(trim(col("StoreRegion")))) \
                     .withColumn("DeviceUsed", upper(trim(col("DeviceUsed"))))

display(df_clean2)

### Data Type Casting
Cast TransactionDate to timestamp, Quantity to integer, Amount and Discount to float

In [0]:
df_clean3 = df_clean2.withColumn("TransactionDate", col("TransactionDate").cast("timestamp")) \
                     .withColumn("Quantity", col("Quantity").cast("int")) \
                     .withColumn("Amount", col("Amount").cast("float")) \
                     .withColumn("Discount", col("Discount").cast("float"))
display(df_clean3)

### Business Rule Validation
Filter transactions: Quantity > 0, Amount > 0, Discount ≤ Amount

In [0]:
# Filter out invalid quantity, negative values, excessive discounts
df_clean4 = df_clean3.filter(
    (col("Quantity") > 0) & 
    (col("Amount") > 0) &
    (col("Discount") <= col("Amount")))
display(df_clean4)

### Deduplication
Remove duplicate transactions by TransactionID

In [0]:
df_clean5 = df_clean4.dropDuplicates(["TransactionID"])

## ⚪ Silver Layer - Data Write

In [0]:
# Write to Silver
df_clean5.write.format("parquet").mode("overwrite").save("/mnt/retail/silver")


In [0]:
silver_df=spark.read.parquet('/mnt/retail/silver/')
display(silver_df)

## 🟡 Gold Layer - Business Analytics

**Objective:** Create aggregated views and business metrics for decision-making

**Setup:**
* Load Silver dataset (cleaned, validated data)
* Register as temporary view `retail_data`
* Enable SQL-based analytics queries

In [0]:
silver_df.createOrReplaceTempView('retail_data')

### 📊 Data Preview
View complete cleaned dataset with all fields

In [0]:
%sql
select * from retail_data

### 📈 Daily Revenue & Purchase Trends
**Business Question:** How does revenue and transaction volume vary day-by-day?

**Metrics:**
* Total revenue per day
* Number of unique transactions per day

In [0]:
%sql
select
    date(TransactionDate) as TransactionDate,
    sum(amount) as total_revenue,
    count(distinct TransactionID) as total_purchase
from retail_data
group by 1

### 💳 Revenue by Payment Type
**Business Question:** Which payment methods generate the most revenue?

**Metrics:**
* Total revenue by payment type (Credit Card, Debit Card, Cash, etc.)


In [0]:
%sql
select sum(amount) total_revenue,PAYMENTTYPE from retail_data
group by 2

### 🏪 Store Performance by Location
**Business Question:** Which store locations are most profitable?

**Metrics:**
* Total revenue by store location

In [0]:
%sql
select sum(amount) total_revenue,STORELOCATION from retail_data
group by 2

### ⭐ Customer Loyalty Impact
**Business Question:** How do different loyalty levels contribute to revenue?

**Metrics:**
* Total revenue by loyalty level (Bronze, Silver, Gold, Platinum)

In [0]:
%sql
select sum(amount) total_revenue,CustomerLoyaltyLevel
 from retail_data
group by 2

### 🛍️ Product Category Sales
**Business Question:** Which product categories drive the most revenue?

**Metrics:**
* Total revenue by product category

In [0]:
%sql
select sum(amount) total_revenue,ProductCategory

 from retail_data
group by 2

### 🌍 Regional Performance Analysis
**Business Question:** How does revenue performance vary across different store regions?

**Metrics:**
* Total revenue by store region
* Total transactions by region
* Unique customers per region

In [0]:
%sql
SELECT 
    StoreRegion,
    SUM(Amount) AS total_revenue,
    COUNT(DISTINCT TransactionID) AS total_transactions,
    COUNT(DISTINCT CustomerID) AS unique_customers
FROM retail_data
GROUP BY StoreRegion
ORDER BY total_revenue DESC